# Pointcloud AE Latent (Mari)
Load trained AE checkpoint, extract latent vectors, and visualize label space colored by cell type.

In [11]:
import json
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import trimesh
from sklearn.decomposition import PCA
import plotly.express as px
import umap

repo_root = Path('..').resolve()
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

from scripts.pointcloud_ae import PointCloudAutoencoder, find_ply_files, sample_points, normalize_points


c:\Users\ljd567\AppData\Local\miniconda3\envs\morphonuc\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
CONFIG_PATH = repo_root / 'configs' / 'pointcloud_ae_mari6.json'
CHECKPOINT_PATH = repo_root / 'models' / 'pointcloud_ae' / 'dgcnn_pointcloud_ae_mari6_best.pth'
METADATA_PATH = repo_root / 'DATA' / 'Maridata' / '6th_data' / 'point_clouds' / 'metadata.csv'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
    config = json.load(f)

print('device:', DEVICE)
print('config:', CONFIG_PATH)
print('checkpoint:', CHECKPOINT_PATH)
print('metadata exists:', METADATA_PATH.exists())


device: cuda
config: D:\Ziwei\Github\MorphoNuc3D\configs\pointcloud_ae_mari6.json
checkpoint: D:\Ziwei\Github\MorphoNuc3D\models\pointcloud_ae\dgcnn_pointcloud_ae_mari6_best.pth
metadata exists: True


In [3]:
model = PointCloudAutoencoder(
    n_points=int(config['n_points']),
    k=int(config['k']),
    emb_dims=int(config['emb_dims']),
    latent_dim=int(config['latent_dim']),
    dropout=float(config.get('dropout', 0.0)),
).to(DEVICE)

ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

print('loaded epoch:', ckpt.get('epoch'))


loaded epoch: 86


In [4]:
data_roots = [repo_root / p for p in config['data_roots']]
ply_paths = find_ply_files(data_roots, config.get('glob_pattern', '*.ply'))
print('point clouds:', len(ply_paths))


point clouds: 12238


In [5]:
def parse_celltype_from_filename(name):
    m = re.match(r'^(.+)_t(\d+)_(\d+)_(.+)\.ply$', name)
    return m.group(4) if m else 'unknown'

def load_one_ply_latent(ply_path, model, n_points, device='cpu'):
    cloud = trimesh.load(ply_path, process=False)
    points = np.asarray(cloud.vertices, dtype=np.float32)
    points = sample_points(points, n_points)
    points = normalize_points(points)

    x = torch.from_numpy(points).unsqueeze(0).to(device=device, dtype=torch.float32)
    with torch.no_grad():
        _, latent = model(x)
    return latent.squeeze(0).cpu().numpy()


In [6]:
latents = []
rows = []
for p in ply_paths:
    latents.append(load_one_ply_latent(p, model, int(config['n_points']), DEVICE))
    rows.append({'ply_path': str(p), 'ply_file': p.name})

latent_arr = np.stack(latents, axis=0)
df = pd.DataFrame(rows)
print('latent shape:', latent_arr.shape)


latent shape: (12238, 128)


In [7]:
if METADATA_PATH.exists():
    meta = pd.read_csv(METADATA_PATH)
    keep_cols = [c for c in ['ply_file', 'cell_type', 'annotation', 'track_id', 'frame', 'label'] if c in meta.columns]
    meta = meta[keep_cols].drop_duplicates(subset=['ply_file'])
    df = df.merge(meta, on='ply_file', how='left')
else:
    df['cell_type'] = df['ply_file'].map(parse_celltype_from_filename)

if 'cell_type' not in df.columns:
    df['cell_type'] = 'unknown'

df['cell_type'] = df['cell_type'].fillna('unknown').astype(str)
df.head()


,ply_path,ply_file,cell_type,annotation,track_id,frame,label
0,D:\Ziwei\Github\MorphoNuc3D\DATA\Maridata\6th_...,1.0_t300_1018_basal.ply,basal,automatic,1.0,300,1018
1,D:\Ziwei\Github\MorphoNuc3D\DATA\Maridata\6th_...,1.0_t300_132_basal.ply,basal,automatic,1.0,300,132
2,D:\Ziwei\Github\MorphoNuc3D\DATA\Maridata\6th_...,1.0_t300_146_basal.ply,basal,automatic,1.0,300,146
3,D:\Ziwei\Github\MorphoNuc3D\DATA\Maridata\6th_...,1.0_t300_181_basal.ply,basal,automatic,1.0,300,181
4,D:\Ziwei\Github\MorphoNuc3D\DATA\Maridata\6th_...,1.0_t301_1039_basal.ply,basal,automatic,1.0,301,1039


In [8]:
pca = PCA(n_components=2, random_state=42)
xy = pca.fit_transform(latent_arr)
df['PC1'] = xy[:, 0]
df['PC2'] = xy[:, 1]

fig = px.scatter(
    df,
    x='PC1',
    y='PC2',
    color='cell_type',
    hover_data=[c for c in ['ply_file', 'track_id', 'frame', 'label', 'annotation'] if c in df.columns],
    title='AE Latent Space (PCA) - Colored by Cell Type',
)
fig.update_traces(marker={'size': 7, 'opacity': 0.85})
fig.show()


In [12]:
umap_xy = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(latent_arr)
df['UMAP1'] = umap_xy[:, 0]
df['UMAP2'] = umap_xy[:, 1]

fig = px.scatter(
    df,
    x='UMAP1',
    y='UMAP2',
    color='cell_type',
    hover_data=[c for c in ['ply_file', 'track_id', 'frame', 'label', 'annotation'] if c in df.columns],
    title='AE Latent Space (UMAP) - Colored by Cell Type',
)
fig.update_traces(marker={'size': 7, 'opacity': 0.85})
fig.show()


c:\Users\ljd567\AppData\Local\miniconda3\envs\morphonuc\lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
